In [20]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import numpy as np
from xgboost import XGBRegressor
from xrfm import xRFM
import seaborn as sns
from sklearn.metrics import mean_squared_error, r2_score
from skopt import BayesSearchCV
from skopt.space import Real, Integer

In [4]:
train.sample(2)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
178,179,20,RL,63.0,17423,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,7,2009,New,Partial,501837
1223,1224,20,RL,89.0,10680,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,10,2006,WD,Normal,137900


In [2]:
train = pd.read_csv('datasets/house_prices/train.csv')
test = pd.read_csv('datasets/house_prices/test.csv')

In [6]:
[col for col in train.columns if col not in test.columns]

['SalePrice']

In [24]:
cat_cols = [col for col in train.columns if train[col].dtype == 'O']
cat_cols.append('MSSubClass')
num_cols = [col for col in train.columns if col not in cat_cols and col != 'SalePrice']

In [17]:
na_cols = [col for col in train.columns if train[col].isna().sum()]
na_num_cols = []
for col in na_cols:
    if col not in cat_cols:
        na_num_cols.append(col)

## Preprocessing data

In [25]:
target = 'SalePrice'
X = train.drop(columns=target)
y = train[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
# ----------------------------
# 1. Your preprocessing
# ----------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_cols)
    ]
)

# ----------------------------
# 2. Full pipeline
# ----------------------------
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1,
        tree_method='hist'   # usually faster on tabular data
    ))
])

# ----------------------------
# 3. K-fold CV
# ----------------------------
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# ----------------------------
# 4. Bayesian search space
# ----------------------------
search_spaces = {

    # Focused XGBoost ranges
    'model__n_estimators': Integer(200, 1200),
    'model__max_depth': Integer(3, 8),
    'model__learning_rate': Real(0.01, 0.15, prior='log-uniform'),
    'model__subsample': Real(0.65, 1.0),
    'model__colsample_bytree': Real(0.65, 1.0),
    'model__min_child_weight': Integer(1, 12),
    'model__gamma': Real(1e-8, 2.0, prior='log-uniform'),
    'model__reg_alpha': Real(1e-8, 2.0, prior='log-uniform'),
    'model__reg_lambda': Real(1e-6, 20.0, prior='log-uniform')
}

# ----------------------------
# 5. Bayesian search
# ----------------------------
opt = BayesSearchCV(
    estimator=pipe,
    search_spaces=search_spaces,
    n_iter=32,   # a bit leaner and more efficient
    scoring='neg_root_mean_squared_error',
    cv=cv,
    n_jobs=-1,
    random_state=42,
    verbose=1,
    refit=True
)

# ----------------------------
# 6. Fit
# ----------------------------
opt.fit(X_train, y_train)

print("Best CV RMSE:", -opt.best_score_)
print("Best params:")
for k, v in opt.best_params_.items():
    print(f"{k}: {v}")

best_model = opt.best_estimator_

# ----------------------------
# 7. Test evaluation
# ----------------------------
y_pred = best_model.predict(X_test)

rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("\nTest RMSE:", rmse)
print("Test R^2:", r2)

Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fi